# Raster-to-vector conversion

This notebook converts the land-cover classification raster into vector polygons and stores the result in a GeoPackage. It is designed to be executed from Visual Studio Code or Jupyter after installing the project dependencies.


In [ ]:
from pathlib import Path

import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape

# Resolve project directories independently of the operating system.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_TIF = DATA_DIR / "Clasificacion_CDMX_recortada.tif"
OUTPUT_GPKG = OUTPUT_DIR / "land_cover_classification.gpkg"
OUTPUT_LAYER = "land_cover_classification"

print(f"Input raster: {INPUT_TIF}")
print(f"Output file: {OUTPUT_GPKG}")


In [ ]:
if not INPUT_TIF.exists():
    raise FileNotFoundError(
        f"Input raster not found: {INPUT_TIF}. "
        "Place it in the data directory using the expected filename."
    )

geometries = []
values = []

with rasterio.open(INPUT_TIF) as src:
    image = src.read(1)
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

    mask = image != nodata if nodata is not None else None

    for geom, value in shapes(image, mask=mask, transform=transform):
        geometries.append(shape(geom))
        values.append(int(value))

land_cover = gpd.GeoDataFrame(
    {"class_id": values},
    geometry=geometries,
    crs=crs,
)

# Remove empty geometries and repair invalid polygon geometries.
land_cover = land_cover[~land_cover.geometry.is_empty].copy()
land_cover["geometry"] = land_cover.geometry.make_valid()

land_cover.to_file(
    OUTPUT_GPKG,
    layer=OUTPUT_LAYER,
    driver="GPKG",
)

print(f"Created {len(land_cover):,} polygons.")
print(f"Saved to: {OUTPUT_GPKG}")
land_cover.head()
